**Week 10**<br />Monday, Nov. 2

# Tagging With Machine Learning

LX 394/694<br />
Coding for Linguists

Sophie Hao


## Announcements

- Welcome back!
- HW 2 submissions will be accepted without penalty until Monday, Nov. 9, at 9 AM.
- HW 3 will be released **next week** on schedule.
- We are now on Chapter 5 of the textbook (Tagging).
- This is the first "real" computational linguistics topic we are covering.
- BUCLD is this week!

## Recap: Tagging

**Tagging** is an NLP task in which each token in a text gets assigned a **tag.**

**Example:** Part of speech (POS) tagging

In [1]:
import nltk

text = nltk.word_tokenize("The quick brown fox jumps over the lazy dog.")
nltk.pos_tag(text)

[('The', 'DT'),
 ('quick', 'JJ'),
 ('brown', 'NN'),
 ('fox', 'NN'),
 ('jumps', 'VBZ'),
 ('over', 'IN'),
 ('the', 'DT'),
 ('lazy', 'JJ'),
 ('dog', 'NN'),
 ('.', '.')]

## Part of Speech Tagging

<br />

| The | quick | brown | fox | jumps | over | the | lazy | dog | . |
|-----|-------|-------|-----|-------|------|-----|------|-----|---|
| DT  | JJ    | NN    | NN  | VBZ   | IN   | DT  | JJ   | NN  | . |

<br />

| Tag | Description | Tag | Description |
|-----|-------------|-----|-------------|
| DT  | Determiner  | VBZ | Verb, 3rd person singular present |
| JJ  | Adjective   | IN  | Preposition or subordinating conjunction |
| NN  | Noun, singular or mass | . | Period |

[See full list of tags](https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html)

## Other Tagging Tasks


**Named Entity Recognition:**

| U.N. | official | Ekeus | heads | for | Baghdad | . |
|------|----------|-------|-------|-----|---------|---|
| ORG  |          | PER   |       |     | LOC     |   |

<span style="color: #cccccc">From <a href="https://aclanthology.org/W03-0419/" style="color: #cccccc">Tjong Kim Sang and De Meulder (2003)</a></span>


**Intent Detection/Slot Filling:**

| Set the temperature | to | 23°C | in | the | living room | . |
|---------------------|----|------|----|-----|-------------|---|
| intent              |    | temperature | | | room        |   |

<span style="color: #cccccc">From <a href="https://arxiv.org/abs/1805.10190" style="color: #cccccc">Coucke et al. (2018)</a></span>

## Recap: How Do You Do POS Tagging?

Simple method: have a `dict` with the POS tag for each token, and just look it up.

```python
class Tagger:
    def __init__():
        self.tags = {"The": "DT", "quick": "JJ", "brown": "NN", ...}

    def get_tag(self, w):
        return self.tags[w] if w in self.tags else w
        
    def tag(self, text):
        return [(w, self.get_tag(w)) for w in text]
```

## Regex Tagger

In [3]:
patterns = [
    (r'.*ing$', 'VBG'),                # gerunds
    (r'.*ed$', 'VBD'),                 # simple past
    (r'.*es$', 'VBZ'),                 # 3rd singular present
    (r'.*ould$', 'MD'),                # modals
    (r'.*\'s$', 'NN$'),                # possessive nouns
    (r'.*s$', 'NNS'),                  # plural nouns
    (r'^-?[0-9]+(\.[0-9]+)?$', 'CD'),  # cardinal numbers
    (r'.*', 'NN')                      # nouns (default)
]

In [4]:
regex = nltk.RegexpTagger(patterns)
text = nltk.word_tokenize("I would have liked to be wearing shoes in the rain.")
regex.tag(text)

[('I', 'NN'),
 ('would', 'MD'),
 ('have', 'NN'),
 ('liked', 'VBD'),
 ('to', 'NN'),
 ('be', 'NN'),
 ('wearing', 'VBG'),
 ('shoes', 'VBZ'),
 ('in', 'NN'),
 ('the', 'NN'),
 ('rain', 'NN'),
 ('.', 'NN')]

## Problems

What are some problems with the simple tagger and the regex tagger?

How might we fix those problems?

## Tagged Corpora

In [5]:
from nltk.corpus import brown

brown.tagged_words()[:10]

[('The', 'AT'),
 ('Fulton', 'NP-TL'),
 ('County', 'NN-TL'),
 ('Grand', 'JJ-TL'),
 ('Jury', 'NN-TL'),
 ('said', 'VBD'),
 ('Friday', 'NR'),
 ('an', 'AT'),
 ('investigation', 'NN'),
 ('of', 'IN')]

## Unigram Tagger

Since tagged corpora exist, we can use them to try and figure out what is the **statistically most likely** tag for each token.

How would we do this?

What are the advantages and disadvantages to this approach?

## Training a Unigram Tagger

In [6]:
import nltk

unigram = nltk.UnigramTagger(brown.tagged_sents())

In [7]:
text = nltk.word_tokenize("The quick brown fox jumps over the lazy dog.")
unigram.tag(text)

[('The', 'AT'),
 ('quick', 'JJ'),
 ('brown', 'JJ'),
 ('fox', 'NN'),
 ('jumps', 'NNS'),
 ('over', 'IN'),
 ('the', 'AT'),
 ('lazy', 'JJ'),
 ('dog', 'NN'),
 ('.', '.')]

## Bigram Tagger

A **bigram** tagger guesses the tag of a token based on the tag of the previous token.

In [9]:
bigram = nltk.BigramTagger(brown.tagged_sents())
bigram.tag(text)

[('The', 'AT'),
 ('quick', 'JJ'),
 ('brown', 'JJ'),
 ('fox', 'NN'),
 ('jumps', 'VBZ'),
 ('over', 'IN'),
 ('the', 'AT'),
 ('lazy', 'JJ'),
 ('dog', 'NN'),
 ('.', '.')]

## Evaluating Taggers

Which method of tagging is the best? We need to **evaluate** our taggers by comparing their performance on a **test set**.

In [10]:
# Train–test split
dataset = brown.tagged_sents(categories="news")
n = int(len(dataset) * .8)
train = dataset[:n]
test = dataset[n:]

In [11]:
# Train
unigram = nltk.UnigramTagger(train)
bigram = nltk.BigramTagger(train)

## Accuracy

Let's measure the **accuracy** of each tagger on our test set: what percentage of tokens are tagged correctly?

In [12]:
# Test
print(f"Regex Tagger:\t{regex.accuracy(test):.3f}")
print(f"Unigram Tagger:\t{unigram.accuracy(test):.3f}")
print(f"Bigram Tagger:\t{bigram.accuracy(test):.3f}")

Regex Tagger:	0.196
Unigram Tagger:	0.803
Bigram Tagger:	0.092


## Error Analysis

The bigram tagger is surprisingly bad. Why?

In [13]:
bigram.tag(brown.sents()[0])

[('The', 'AT'),
 ('Fulton', 'NP-TL'),
 ('County', 'NN-TL'),
 ('Grand', 'JJ-TL'),
 ('Jury', 'NN-TL'),
 ('said', 'VBD'),
 ('Friday', 'NR'),
 ('an', 'AT'),
 ('investigation', 'NN'),
 ('of', 'IN'),
 ("Atlanta's", 'NP$'),
 ('recent', 'JJ'),
 ('primary', 'NN'),
 ('election', 'NN'),
 ('produced', 'VBD'),
 ('``', '``'),
 ('no', 'AT'),
 ('evidence', 'NN'),
 ("''", "''"),
 ('that', 'CS'),
 ('any', 'DTI'),
 ('irregularities', 'NNS'),
 ('took', 'VBD'),
 ('place', 'NN'),
 ('.', '.')]

In [15]:
# What is the most common incorrect tag?
from nltk.probability import FreqDist
error_fd = FreqDist()

for sent in test:
    words, tags = zip(*sent)
    _, pred_tags = zip(*bigram.tag(words))
    error_fd.update([pt for t, pt in zip(tags, pred_tags) if t != pt])
    
error_fd.most_common(5)

[(None, 18743), ('IN', 16), ('CS', 13), ('TO', 12), ('RB', 9)]

In [ ]:
# Train some backoff taggers
unigram_backoff = nltk.UnigramTagger(train, backoff=regex)
bigram_backoff = nltk.BigramTagger(train, backoff=unigram_backoff)

In [ ]:
# Evaluate
print(f"Regex Tagger:\t {regex.accuracy(test):.3f}")
print(f"Unigram Tagger:\t {unigram.accuracy(test):.3f}")
print(f"Bigram Tagger:\t {bigram.accuracy(test):.3f}\n")

print(f"Unigram Backoff: {unigram_backoff.accuracy(test):.3f}")
print(f"Bigram Backoff:\t {bigram_backoff.accuracy(test):.3f}\n")

## Next Time

Machine Learning, Continued